# Preprocessing — Jalur **SVM + TF-IDF**

Notebook **self-contained** (semua kode preprocessing ada di sini, tanpa import `src/`).
Cocok dijalankan lokal maupun di Google Colab.

Jalur SVM memakai **cleaning agresif** karena TF-IDF hanya menghitung frekuensi kata
(bag-of-words), tidak paham konteks/urutan. Empat tahap:
1. **clean_for_svm** — lowercase, buang URL/mention/hashtag/emoji/angka/tanda baca.
2. **normalize_slang** — `gak→tidak`, `sdh→sudah`, dst.
3. **remove_stopword** — buang kata fungsi yang jadi noise.
4. **stemming (Sastrawi)** — `kebohongan→bohong` (kurangi sparsity).

> **Split identik** dengan notebook IndoBERT: urut `comment_id` + `seed=42`, split
> dilakukan SEBELUM preprocessing → test set kedua model persis sama (perbandingan adil).

## 0. Install dependency

In [ ]:
!pip -q install PySastrawi pyarrow scikit-learn

## 1. Muat data berlabel (balanced 3.000)

In [ ]:
import os, pandas as pd

# Path lokal default; kalau tak ada (mis. di Colab) -> minta upload.
INPUT = "outputs/labeling/balanced_1000.csv"
if not os.path.exists(INPUT):
    try:
        from google.colab import files
        up = files.upload()
        INPUT = next(iter(up))
    except Exception:
        raise FileNotFoundError(
            "balanced_1000.csv tidak ditemukan. Taruh di folder kerja "
            "atau upload saat diminta (Colab)."
        )

df = pd.read_csv(INPUT, encoding="utf-8-sig")
LABELS = ["Negatif", "Netral", "Positif"]          # urutan menentukan id: Neg=0, Net=1, Pos=2
df = df[df["label"].isin(LABELS)].copy()
print(f"{len(df)} baris berlabel dimuat dari {INPUT}")
print(df["label"].value_counts().reindex(LABELS).to_string())

## 2. Fungsi preprocessing SVM (inline)

In [ ]:
import re
from typing import List

SLANG_DICT = {'aja': 'saja',
 'bagus': 'bagus',
 'baguss': 'bagus',
 'bang': '',
 'banget': 'sangat',
 'bener': 'benar',
 'bgt': 'sangat',
 'blm': 'belum',
 'blom': 'belum',
 'bngt': 'sangat',
 'bro': '',
 'buat': 'untuk',
 'bwt': 'untuk',
 'channel': 'saluran',
 'cuma': 'hanya',
 'dah': 'sudah',
 'deh': '',
 'dg': 'dengan',
 'dgn': 'dengan',
 'dong': '',
 'dr': 'dari',
 'dri': 'dari',
 'elu': 'kamu',
 'enggak': 'tidak',
 'g': 'tidak',
 'ga': 'tidak',
 'gak': 'tidak',
 'gakk': 'tidak',
 'gan': '',
 'gila': 'gila',
 'gilaa': 'gila',
 'gk': 'tidak',
 'gua': 'saya',
 'gue': 'saya',
 'gw': 'saya',
 'haha': '',
 'hehe': '',
 'hihi': '',
 'iya': 'ya',
 'iyaa': 'ya',
 'jelek': 'jelek',
 'jelekk': 'jelek',
 'jg': 'juga',
 'jga': 'juga',
 'kak': '',
 'kalo': 'kalau',
 'kalu': 'kalau',
 'kan': '',
 'karna': 'karena',
 'keren': 'keren',
 'kerennn': 'keren',
 'kl': 'kalau',
 'klo': 'kalau',
 'klu': 'kalau',
 'kok': '',
 'komen': 'komentar',
 'konten': 'konten',
 'krn': 'karena',
 'kwkw': '',
 'lah': '',
 'lg': 'lagi',
 'lgi': 'lagi',
 'like': 'suka',
 'lo': 'kamu',
 'lu': 'kamu',
 'makasi': 'terima kasih',
 'makasih': 'terima kasih',
 'mantap': 'mantap',
 'mantep': 'mantap',
 'min': '',
 'mksh': 'terima kasih',
 'ngga': 'tidak',
 'nggak': 'tidak',
 'nih': '',
 'ntar': 'nanti',
 'ok': 'oke',
 'org': 'orang',
 'q': 'saya',
 'sdh': 'sudah',
 'share': 'bagikan',
 'sih': '',
 'smgt': 'semangat',
 'sub': 'berlangganan',
 'subscribe': 'berlangganan',
 'sy': 'saya',
 'tak': 'tidak',
 'tar': 'nanti',
 'tau': 'tahu',
 'tdk': 'tidak',
 'tks': 'terima kasih',
 'tp': 'tapi',
 'tpi': 'tapi',
 'trs': 'terus',
 'trus': 'terus',
 'udah': 'sudah',
 'upload': 'unggah',
 'utk': 'untuk',
 'wkwk': '',
 'wkwkwk': '',
 'xD': '',
 'xd': '',
 'yg': 'yang',
 'yng': 'yang'}

STOPWORDS_ID = {'ada',
 'adalah',
 'agar',
 'akan',
 'amat',
 'anda',
 'antara',
 'apa',
 'atas',
 'atau',
 'bagaimana',
 'baru',
 'bawah',
 'beberapa',
 'belum',
 'besar',
 'bisa',
 'bukan',
 'dalam',
 'dan',
 'dari',
 'dengan',
 'di',
 'dia',
 'dua',
 'ia',
 'ini',
 'itu',
 'jarang',
 'jika',
 'juga',
 'kadang',
 'kami',
 'kamu',
 'karena',
 'ke',
 'kecil',
 'ketika',
 'kita',
 'lagi',
 'lain',
 'lama',
 'lebih',
 'luar',
 'maka',
 'masih',
 'mengapa',
 'mereka',
 'namun',
 'oleh',
 'pada',
 'pernah',
 'pun',
 'saat',
 'saja',
 'sama',
 'sangat',
 'satu',
 'saya',
 'sebelum',
 'sedang',
 'sejak',
 'sekali',
 'selalu',
 'selama',
 'semua',
 'sering',
 'setelah',
 'setiap',
 'sudah',
 'supaya',
 'tapi',
 'telah',
 'tetapi',
 'tidak',
 'tiga',
 'untuk',
 'ya',
 'yang'}

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_HASHTAG = re.compile('#\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_NUMBER = re.compile('\\b\\d+\\b')
_RE_NON_ALPHA = re.compile('[^a-z\\s]')
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_svm(text: str) -> str:
    """Cleaning agresif untuk jalur SVM + TF-IDF."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_HASHTAG.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = _RE_NUMBER.sub(" ", t)
    t = _RE_NON_ALPHA.sub(" ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def normalize_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    """Ganti kata slang/informal dengan bentuk baku."""
    if not text:
        return ""
    tokens = text.split()
    normalized = [slang_dict.get(tok, tok) for tok in tokens]
    result = " ".join(tok for tok in normalized if tok)
    return _RE_MULTI_SPACE.sub(" ", result).strip()


def tokenize(text: str) -> List[str]:
    """Tokenisasi sederhana berdasarkan whitespace."""
    return text.split() if text else []


def remove_stopwords(tokens: List[str], stopwords: set = STOPWORDS_ID) -> List[str]:
    """Hapus stopword dari daftar token."""
    return [tok for tok in tokens if tok not in stopwords and len(tok) > 1]


def preprocess_svm_python(text: str) -> str:
    """Pipeline SVM tanpa stemming (stemming dilakukan terpisah via Sastrawi UDF)."""
    cleaned = clean_for_svm(text)
    normalized = normalize_slang(cleaned)
    tokens = tokenize(normalized)
    tokens = remove_stopwords(tokens)
    return " ".join(tokens)

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
_stemmer = StemmerFactory().create_stemmer()   # mahal -> buat sekali

def make_text_svm(text: str) -> str:
    """Pipeline final jalur SVM: clean+slang+stopword lalu stemming."""
    pre = preprocess_svm_python(text or "")
    return _stemmer.stem(pre) if pre else pre

## 3. Lihat transformasi langkah-demi-langkah

In [ ]:
def trace_svm(text: str) -> None:
    print(f"RAW             : {text!r}")
    s1 = clean_for_svm(text)
    s2 = normalize_slang(s1)
    s3 = " ".join(remove_stopwords(tokenize(s2)))
    s4 = _stemmer.stem(s3)
    print(f"1. clean_for_svm  : {s1!r}")
    print(f"2. normalize_slang: {s2!r}")
    print(f"3. remove_stopword: {s3!r}")
    print(f"4. stem (FINAL)   : {s4!r}")

trace_svm("Persidangan belum dilaksanakan kok sdh ngajuin ahli, kalo gini rakyat dibikin binggung")

## 4. Terapkan ke semua data + split identik (70/20/10)

In [ ]:
from sklearn.model_selection import train_test_split

LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}

# --- SPLIT IDENTIK utk SVM & IndoBERT ---
# Urutkan comment_id + seed tetap -> kedua notebook menghasilkan train/val/test
# yang PERSIS SAMA (test set sama -> perbandingan adil). Split dilakukan SEBELUM
# preprocessing, jadi tak terpengaruh perbedaan cleaning antar jalur.
df = df.sort_values("comment_id").reset_index(drop=True)
df["label_id"] = df["label"].map(LABEL2ID)

SEED = 42
tmp, df_test = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=SEED)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=SEED)

splits = {}
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    part = part.copy()
    part["svm"] = part["text"].astype(str).map(make_text_svm)      # preprocessing jalur ini
    before = len(part)
    part = part[part["svm"].str.len() > 0]                          # buang yang kosong stlh cleaning
    if before - len(part):
        print(f"[{name}] {before-len(part)} baris dibuang (kosong stlh preprocessing)")
    splits[name] = part.reset_index(drop=True)

for name, part in splits.items():
    dist = " ".join(f"{k}={v}" for k, v in part["label"].value_counts().reindex(LABELS).items())
    print(f"[{name}] n={len(part)} | {dist}")

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)
splits["train"][["text", "svm", "label"]].head(8)

## 5. Simpan split siap-model

In [ ]:
OUTDIR = "data/processed/svm"
os.makedirs(OUTDIR, exist_ok=True)
cols = ["comment_id", "video_id", "text", "svm", "label", "label_id"]
for name, part in splits.items():
    keep = [c for c in cols if c in part.columns]
    part[keep].to_parquet(f"{OUTDIR}/{name}.parquet", index=False)
print("Tersimpan ke", OUTDIR, "->", list(splits))

# (Opsional, Colab) unduh hasil:
# from google.colab import files
# for name in splits: files.download(f"{OUTDIR}/{name}.parquet")

Output: `data/processed/svm/{train,val,test}.parquet` (kolom **`svm`** = teks bersih).
**Lanjut:** notebook training SVM (TF-IDF + LinearSVC).